<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>Cómo funcionan las redes neuronales</b></font></h1>
<h2><b>Notebook 3: </b>Hot dog or not</h2>

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models

import utils
from utils import CLASS_NAMES, DEVICE, EPOCHS, WEIGHTS

utils.set_seeds(42)

----

<br>

## **Parte 0:**

<br>

<h1 style="text-align: center; font-size: 3em; line-height: 1.4;"><a href="https://www.youtube.com/watch?v=tWwCK95X6go" target="_blank">Contexto 🌭</a></h1>

<br>

----

<br>

## **Parte 1:** Configuración

In [ ]:
def load(split, size):
    folder = datasets.ImageFolder(f"data/hotdogs_or_not/{split}",
                                  WEIGHTS.transforms(resize_size=size, crop_size=size))
    images, classes = zip(*folder)
    X = torch.stack(images).to(DEVICE)
    y = (torch.tensor(classes) == folder.class_to_idx["hot_dog"]).float().unsqueeze(1).to(DEVICE)
    return DataLoader(TensorDataset(X, y), batch_size=32, shuffle=(split == "train"))


train_loader, val_loader = load("train", 64), load("test", 64)

images, labels = train_loader.dataset.tensors
sample = torch.randperm(len(images))[:12]
utils.plot_sample(images[sample], [CLASS_NAMES[int(label)] for label in labels[sample]],
                  title="Doce fotos de entrenamiento")

In [ ]:
def train(model, loader, epochs=EPOCHS):
    loss_function = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = []

    for epoch in range(epochs):
        model.train()
        total = 0
        for X, y in loader:
            optimizer.zero_grad()
            loss = loss_function(model(X), y)
            loss.backward()
            optimizer.step()
            total += loss.item()
        history.append(total / len(loader))

    return history


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true = torch.cat([y for _, y in loader]).cpu()
    y_pred = torch.cat([(model(X) > 0).float() for X, _ in loader]).cpu()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "matrix": confusion_matrix(y_true, y_pred),
    }

----

<br>

## **Parte 2:** Optuna escoge la CNN

In [ ]:
def create_cnn(trial):
    layers, channels, side = [], 3, 64
    for i in range(trial.suggest_int("n_layers", 2, 4)):
        filters = trial.suggest_int(f"filters_{i}", 16, 128)
        layers += [nn.Conv2d(channels, filters, 3, padding=1), nn.BatchNorm2d(filters), nn.ReLU(), nn.MaxPool2d(2)]
        channels, side = filters, side // 2

    neurons = trial.suggest_int("neurons", 64, 256)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    layers += [nn.Flatten(), nn.Dropout(dropout), nn.Linear(channels * side * side, neurons), nn.ReLU(),
               nn.Dropout(dropout), nn.Linear(neurons, 1)]
    return nn.Sequential(*layers).to(DEVICE)


def objective(trial):
    model = create_cnn(trial)
    train(model, train_loader)
    return evaluate(model, val_loader)["accuracy"]


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)

print("mejor accuracy:", round(study.best_value, 3))
print("hiperparámetros:", study.best_params)

In [ ]:
cnn = create_cnn(optuna.trial.FixedTrial(study.best_params))
history = train(cnn, train_loader)

utils.plot_metrics({"entrenamiento": history}, title="La mejor CNN, época a época",
                   y_label="pérdida (BCE)")

results_cnn = evaluate(cnn, val_loader)
utils.plot_results(results_cnn, title="Mejor CNN")

In [ ]:
images, _ = val_loader.dataset.tensors
sample = torch.randperm(len(images))[:12]

with torch.no_grad():
    probabilities = torch.sigmoid(cnn(images[sample]))

utils.plot_predictions(images[sample], probabilities, title="Hot dog or not")

----

<br>

## **Parte 4:** ¿Hacía falta una CNN?

In [ ]:
logistic = nn.Sequential(nn.Flatten(), nn.Linear(3 * 64 * 64, 1)).to(DEVICE)
train(logistic, train_loader)

results_logistic = evaluate(logistic, val_loader)
utils.plot_results(results_logistic, title="Regresión logística")

----

<br>

## **Parte 5:** Transfer learning

In [ ]:
train_loader_224, val_loader_224 = load("train", 224), load("test", 224)

resnet = models.resnet18(weights=WEIGHTS)
resnet.requires_grad_(False)                      
resnet.fc = nn.Linear(resnet.fc.in_features, 1)     
resnet = resnet.to(DEVICE)

train(resnet, train_loader_224)

results_transfer = evaluate(resnet, val_loader_224)
utils.plot_results(results_transfer, title="Transfer learning")

In [ ]:
utils.plot_comparison({"regresión logística": results_logistic, "CNN (Optuna)": results_cnn,"transfer learning": results_transfer}, title="Conjunto de validación")

for name, model in [("regresión logística", logistic), ("CNN (Optuna)", cnn), ("transfer learning", resnet)]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"{name:<20} {trainable:>10,} parámetros entrenados de {total:>11,}")

-----

<br>

### **Siguiente notebook:**

[`04_tictactoe.ipynb`](04_tictactoe.ipynb)

### <font color="#29c4d9">**Notebook 3 listo.**</font>

<br>

---

<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>